## Native S3 Access Using `s3fs`

This notebook demonstrates secure and authenticated interaction with AWS S3 using `s3fs`.  
We use both Python-style I/O (`fs.open`) and pandas-style loading (`s3://...`), with bucket checks, file reads/writes, and size introspection.

AWS credentials must be configured in `~/.aws/credentials` for `anon=False` to work.

This notebook demonstrates native S3 interaction using the `s3fs` library. It covers:
- Bucket connection
- Listing files
- Reading and writing files
- Pandas integration via `s3://` URIs

These operations follow the recommended practices from the [s3fs documentation](https://s3fs.readthedocs.io/en/latest/#examples).


In [ ]:
import s3fs
import pandas as pd

# ✅ Step 1: Initialize the S3 connection
fs = s3fs.S3FileSystem(anon=False)

# ✅ Define bucket name first
bucket_name = 'bitcoin-timeseries-data-kv'

# ✅ Verifying if bucket exists or is accessible
if not fs.exists(bucket_name):
    raise ValueError(f"❌ Bucket {bucket_name} not accessible or does not exist.")

print("✅ Connected to S3 bucket.")

### 📂 List Files in the S3 Bucket
We can list files stored in the bucket using `fs.ls()`.


In [ ]:
files = fs.ls(bucket_name)
print("✅ Files in bucket:", files)


### 📄 Read First Few Bytes of a File
Use `fs.open()` in binary read mode (`'rb'`) to read any file.


In [ ]:
file_path = f'{bucket_name}/bitcoin_prices.csv'
# Optional: Check if file exists before reading
if not fs.exists(file_path):
    raise FileNotFoundError(f"❌ File not found: {file_path}")

with fs.open(file_path, 'rb') as f:
    content = f.read(200)

print("✅ First 200 bytes of file content:\n")
print(content.decode('utf-8'))


### ☁️ Write a Text File to S3
We can write text or binary content using `fs.open()` in write mode.


In [ ]:
test_path = f'{bucket_name}/test_upload.txt'
with fs.open(test_path, 'wb') as f:
    f.write(b'This is a test upload to S3 using s3fs.')

print("✅ File uploaded to S3 successfully!")


### 📊 Load CSV from S3 into Pandas using `fs.open()`
This method gives fine-grained control and allows streaming large files.


In [ ]:
import pandas as pd

csv_path = f'{bucket_name}/bitcoin_prices.csv'
with fs.open(csv_path, 'rb') as f:
    df = pd.read_csv(f)

print("✅ Dataframe loaded from S3 (via fs.open):")
df.head()


### 🔄 Alternative: Load CSV Using `s3://` URI
This is a more concise syntax and often works well with small/medium files.


In [ ]:
df_s3 = pd.read_csv(f's3://{bucket_name}/bitcoin_prices.csv', storage_options={'anon': False})
print("✅ Loaded using s3:// path:")
df_s3.head()


### 📦 Explore Files in the Bucket
Use `.ls()` to list and `.du()` to check file sizes.


In [ ]:
print("✅ Bucket Contents:")
print(fs.ls(bucket_name))

print("\n✅ File Size:")
print(fs.du(f'{bucket_name}/bitcoin_prices.csv'))


## ✅ Summary

This notebook demonstrates native S3 access using `s3fs`:
- Secure and authenticated using AWS credentials
- Fully integrated with Python file operations and Pandas
- Enables cloud-based workflows with no reliance on local storage

This aligns with modern data engineering and the tutorial structure required for DATA605.
